# Clase 07 — Conexión entre Python y SQLite

En este cuaderno vas a crear la base de datos `ventas.db`, conectarte desde Python y traer una tabla a un DataFrame de Pandas. Ejecutá las celdas en orden, de arriba hacia abajo.

## 1. Importar las librerías

`sqlite3` viene incluido con Python. `pandas` nos permite leer el resultado de una consulta SQL como una tabla.

In [1]:
import sqlite3
import pandas as pd

## 2. Crear la conexión

Si `ventas.db` no existe, SQLite la crea en esta misma carpeta. Si ya existe, se conecta a ella.

In [2]:
conexion = sqlite3.connect("ventas.db")
cursor = conexion.cursor()

print("Conexión creada con ventas.db")

Conexión creada con ventas.db


## 3. Crear la tabla `productos`

Esta celda define la estructura de la tabla. `IF NOT EXISTS` evita un error si volvés a ejecutar el cuaderno.

In [3]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS productos (
        id INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        categoria TEXT NOT NULL,
        precio REAL NOT NULL,
        stock INTEGER NOT NULL
    )
""")
conexion.commit()

print("Tabla productos lista.")

Tabla productos lista.


## 4. Cargar datos de ejemplo

Los datos se insertan solamente si la tabla está vacía. Así podés volver a ejecutar la celda sin duplicar productos.

In [4]:
cantidad_productos = cursor.execute("SELECT COUNT(*) FROM productos").fetchone()[0]

if cantidad_productos == 0:
    productos = [
        ("Notebook", "tech", 320000, 5),
        ("Mouse", "tech", 8500, 42),
        ("Escritorio", "muebles", 85000, 3),
        ("Silla", "muebles", 45000, 8),
        ("Monitor", "tech", 210000, 0),
        ("Auriculares", "tech", 32000, 18),
        ("Lámpara", "muebles", 12000, 25),
    ]

    cursor.executemany("""
        INSERT INTO productos (nombre, categoria, precio, stock)
        VALUES (?, ?, ?, ?)
    """, productos)
    conexion.commit()
    print(f"Se cargaron {len(productos)} productos.")
else:
    print(f"La tabla ya tiene {cantidad_productos} productos; no se agregaron duplicados.")

Se cargaron 7 productos.


## 5. Traer los datos a Python

`pd.read_sql()` ejecuta la consulta SQL y devuelve un DataFrame. Desde aquí podés usar las herramientas de Pandas que ya conocés.

In [5]:
df_productos = pd.read_sql("SELECT * FROM productos", conexion)
df_productos

,id,nombre,categoria,precio,stock
0,1,Notebook,tech,320000.0,5
1,2,Mouse,tech,8500.0,42
2,3,Escritorio,muebles,85000.0,3
3,4,Silla,muebles,45000.0,8
4,5,Monitor,tech,210000.0,0
5,6,Auriculares,tech,32000.0,18
6,7,Lámpara,muebles,12000.0,25


## 6. Primera consulta

Esta consulta trae solamente los productos que tienen stock y los ordena del más caro al más barato.

In [6]:
consulta = """
    SELECT nombre, categoria, precio, stock
    FROM productos
    WHERE stock > 0
    ORDER BY precio DESC
"""

df_disponibles = pd.read_sql(consulta, conexion)
df_disponibles

,nombre,categoria,precio,stock
0,Notebook,tech,320000.0,5
1,Escritorio,muebles,85000.0,3
2,Silla,muebles,45000.0,8
3,Auriculares,tech,32000.0,18
4,Lámpara,muebles,12000.0,25
5,Mouse,tech,8500.0,42


## 7. Verificar la estructura

Antes de la práctica, confirmamos qué tabla existe y cuáles son sus columnas.

In [7]:
tablas = pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", conexion)
estructura = pd.read_sql("PRAGMA table_info(productos)", conexion)

print("Tablas disponibles:")
display(tablas)

print("Estructura de productos:")
display(estructura)

Tablas disponibles:


,name
0,productos


Estructura de productos:


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,nombre,TEXT,1,None,0
2,2,categoria,TEXT,1,None,0
3,3,precio,REAL,1,None,0
4,4,stock,INTEGER,1,None,0


## 8. Cerrar la conexión

Ejecutá esta última celda cuando termines de trabajar con la base. Si volvés a ejecutar una celda anterior, creá otra vez la conexión primero.

In [8]:
conexion.close()
print("Conexión cerrada.")

Conexión cerrada.
